### Understanding GIL and Multi-Threading Performance

In [83]:
def fibonacci(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)

def helper(n):
    result = fibonacci(n)
    print(f"Fibonacci of number {n} is {result}")

import time
start_time = time.time()
helper(20)
helper(30)
helper(35)
print(f"Sequential processing took: {time.time() - start_time} seconds.")

start_time = time.time()
import threading
threads = [threading.Thread(target=helper,args=(num,)) for num in [20, 30, 35]]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(f"Multi threading took: {time.time() - start_time} seconds.")

Fibonacci of number 20 is 6765
Fibonacci of number 30 is 832040
Fibonacci of number 35 is 9227465
Sequential processing took: 1.1203179359436035 seconds.
Fibonacci of number 20 is 6765
Fibonacci of number 30 is 832040
Fibonacci of number 35 is 9227465
Multi threading took: 1.090663194656372 seconds.


### MultiProcessing

In [90]:
from multiprocessing import Process
import time

def fibonacci(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)

def helper(n):
    result = fibonacci(n)
    return f"Fibonacci of number {n} is {result}"

if __name__ == "__main__":
    processes = [Process(target=helper,args=(num,)) for num in [20, 30, 35]]
    start_time = time.time()
    for process in processes:
        process.start()
    for process in processes:
        process.join()
    print(f"Multi processing took: {time.time() - start_time} seconds.")

Multi processing took: 0.06461477279663086 seconds.


Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=82, pipe_handle=85)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/homebrew/Cellar/python@3.13/3.13.1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
AttributeError: Can't get attribute 'helper' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=82, pipe_handle=85)
                                

### GIL and I/O-bound Tasks
When a thread is performing an I/O task, GIL is released. This is not the case for CPU intensive tasks. Thus, we will see performance benefit when using multi-threading for I/O tasks.

In [95]:
def download_file(n):
    time.sleep(n * 2)
    print(f"Downloaded file: {n}")


import time
start_time = time.time()
for i in range(3):
    download_file(i)
print(f"Sequential processing took {time.time() - start_time}")

import threading
start_time = time.time()
threads = [threading.Thread(target=download_file, args=(i,)) for i in range(3)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(f"Multi-threading took {time.time() - start_time}")

# from multiprocessing import Process
# start_time = time.time()
# processes = [Process(target=download_file, args=(i,)) for i in range(3)]
# for process in processes:
#     process.start()
# for process in processes:
#     process.join()
# print(f"Multi-processing took {time.time() - start_time}")

Downloaded file: 0
Downloaded file: 1
Downloaded file: 2
Sequential processing took 6.006895065307617
Downloaded file: 0
Downloaded file: 1
Downloaded file: 2
Multi-threading took 4.006070137023926


### Measuring GIL Contention with Multiple Threads

In [ ]:
from threading import Thread

def fibonacci(n):
    if n == 0:
        return 0
    if n == 1:
        return 1
    return fibonacci(n - 1) + fibonacci(n - 2)

def helper(n):
    result = fibonacci(n)
    print(f"Fibonacci of number {n} is {result}")

import time
for num_threads in [1, 2, 4, 8]:
    start_time = time.time()
    print(f"------------- {num_threads} --------------")
    threads = [Thread(target=helper, args=(6 * i,)) for i in range(1, num_threads + 1)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    print(f"{num_threads} thread execution took {time.time() - start_time} seconds")

------------- 1 --------------
Fibonacci of number 6 is 8
1 thread execution took 0.11157107353210449 seconds
------------- 2 --------------
Fibonacci of number 6 is 8
Fibonacci of number 12 is 144
2 thread execution took 0.19437909126281738 seconds
------------- 4 --------------
Fibonacci of number 6 is 8
Fibonacci of number 12 is 144
Fibonacci of number 18 is 2584
Fibonacci of number 24 is 46368
4 thread execution took 0.2136549949645996 seconds
------------- 8 --------------
Fibonacci of number 6 is 8
Fibonacci of number 12 is 144
Fibonacci of number 18 is 2584
Fibonacci of number 24 is 46368
Fibonacci of number 30 is 832040
Fibonacci of number 36 is 14930352


KeyboardInterrupt: 

Fibonacci of number 42 is 267914296


### Using C Extensions (numpy or ctypes) to Bypass the GIL

In [3]:
import numpy as np
def matrix_multiplication(size):
    A = np.random.rand(size, size)
    B = np.random.rand(size, size)
    return np.dot(A, B) # NumPy releases the GIL here

from threading import Thread
import time
for num_threads in [1, 2, 4, 8]:
    threads = [Thread(target=matrix_multiplication, args=(num_threads * 50,)) 
               for _ in range(1, num_threads + 1)]
    start_time = time.time()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    print(f"{num_threads} completed in {time.time() - start_time}")

1 completed in 0.02915787696838379
2 completed in 0.08487606048583984
4 completed in 0.8130409717559814
8 completed in 11.742515802383423


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 20.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
